Assignment 4 – A Cognitive FAQ System Using Pandas (Nova 2.0)

Q1: Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas 
DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your 
own roll number digits as follows: 
• Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account", 
"general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category (e.g. 
if d%3 gives "account", write a question like “how do I update my registered mobile number”).
• # Example roll number ...23 -> digits 2, 3
• # digit 2 -> category[2 % 3] = general
• # digit 3 -> category[3 % 3] = billing

In [23]:
import pandas as pd
import numpy as np
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

roll_number = 1024170422

last_two_digits = [int(digit) for digit in str(roll_number)[-2:]]

categories = ["billing", "account", "general"]

category_templates = {
    "billing": {
        "question": "where can i check my billing details",
        "answer": "You can check your billing details in the billing section.",
        "keywords": "billing details charges"
    },
    "account": {
        "question": "how can i update my profile",
        "answer": "You can update your profile from the account settings.",
        "keywords": "profile update account"
    },
    "general": {
        "question": "where is lab L301",
        "answer": "Go to CSED 3rd Floor.",
        "keywords": "lab location L301"
    }
}

personalized_entries = []

for digit in last_two_digits:
    category = categories[digit % 3]

    entry = {
        "question": category_templates[category]["question"],
        "answer": category_templates[category]["answer"],
        "keywords": category_templates[category]["keywords"],
        "category": category
    }

    personalized_entries.append(entry)



personalized_entries.extend(fixed_entries)
personalized_entries


[{'question': 'where is lab L301',
  'answer': 'Go to CSED 3rd Floor.',
  'keywords': 'lab location L301',
  'category': 'general'},
 {'question': 'where is lab L301',
  'answer': 'Go to CSED 3rd Floor.',
  'keywords': 'lab location L301',
  'category': 'general'},
 {'question': 'what is the annual fee',
  'answer': 'The annual fee is Rs 500.',
  'keywords': 'fee cost price charge',
  'category': 'billing'},
 {'question': 'how to reset password',
  'answer': 'Go to Settings > Reset Password.',
  'keywords': 'password reset login',
  'category': 'account'},
 {'question': 'what are your working hours',
  'answer': 'We are open 9 AM to 5 PM.',
  'keywords': 'hours timing open time',
  'category': 'general'},
 {'question': 'how can i pay the fee',
  'answer': 'You can pay via UPI, card, or net banking.',
  'keywords': 'pay payment upi fee',
  'category': 'billing'}]

In [24]:
df = pd.DataFrame(personalized_entries)

Q2: Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching 
entries ranked by confidence

In [34]:
def score_query(query,df):
  keywords = [sent.lower().split() for sent in df['keywords']]
  actual = query.lower().split()
  order = []
  for i in range(len(keywords)):
    cnt = 0
    for w in actual:
      if(w in keywords[i]):
        cnt+=1
    order.append(cnt)
  sorted_order = np.argsort(order)[::-1]
  ans =[]
  for i in range(len(order)):
    ans.append(df['question'].iloc[sorted_order[i]])
  return ans

In [35]:
print("enter a query")
query = input()
query
sorted_FAQ = score_query(query,df)
print("sorted query")
print(sorted_FAQ)

enter a query
sorted query
['how to reset password', 'how can i pay the fee', 'what are your working hours', 'what is the annual fee', 'where is lab L301', 'where is lab L301']


Q3: Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it 
using the category of one of the personalized entries from Q1, and print the result.

In [ ]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"]

In [37]:
category_name = personalized_entries[0]["category"]
result = same_category(category_name, df)

print("Questions in category:", category_name)
print(result)

Questions in category: general
['where is lab L301', 'where is lab L301', 'what are your working hours']


Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and 
save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv.

In [38]:
index = 0
new_keyword = input("Enter a new keyword: ")
df.loc[index, "keywords"] = df.loc[index, "keywords"] + " " + new_keyword

filename = str(roll_number) + "_faq_data.csv"

df.to_csv(filename, index=False)

print("Updated DataFrame:")
print(df)

print("File saved as:", filename)

Updated DataFrame:
                      question                                      answer  \
0            where is lab L301                       Go to CSED 3rd Floor.   
1            where is lab L301                       Go to CSED 3rd Floor.   
2       what is the annual fee                   The annual fee is Rs 500.   
3        how to reset password            Go to Settings > Reset Password.   
4  what are your working hours                   We are open 9 AM to 5 PM.   
5        how can i pay the fee  You can pay via UPI, card, or net banking.   

                   keywords category  
0  lab location L301 annual  general  
1         lab location L301  general  
2     fee cost price charge  billing  
3      password reset login  account  
4    hours timing open time  general  
5       pay payment upi fee  billing  
File saved as: 1024170422_faq_data.csv


Q5: Using groupby, print how many FAQ entries you have per category.

In [39]:
category_counts = df.groupby("category").size()

print("FAQ entries per category:")
print(category_counts)

FAQ entries per category:
category
account    1
billing    2
general    3
dtype: int64


Q6: Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one 
— it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that 
produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [40]:
def score_query(query, df):
    keywords = [sent.lower().split() for sent in df["keywords"]]
    actual = query.lower().split()
    order = []

    for i in range(len(keywords)):
        cnt = 0
        for w in actual:
            if w in keywords[i]:
                cnt += 1
        order.append(cnt)
    max_score = max(order)
    ans = []

    for i in range(len(order)):
        if order[i] == max_score and max_score > 0:
            ans.append({
                "question": df["question"].iloc[i],
                "score": order[i]
            })

    return ans

In [42]:
print("Tie:")
query = "fee"
result = score_query(query, df)
print(result)

print("Non-tie :")
query = "payment upi"
result = score_query(query, df)
print(result)

Tie:
[{'question': 'what is the annual fee', 'score': 1}, {'question': 'how can i pay the fee', 'score': 1}]
Non-tie :
[{'question': 'how can i pay the fee', 'score': 2}]
